# Alpha Connectome, sweep 4: better teachers, out-of-fold teachers, student recipes

Sweep 3 showed distillation works: a 128-unit student (0.6821 valid) nearly matched its 4-model teacher (0.6826)
at single-model speed. This run looks for a better teacher and a better student recipe. About 2-2.5 h on an A100.

* **Phase 1:** 8 out-of-fold teacher models (2 configs x 4 folds, each trained on 3/4 of the data).
* **Phase 2:** four teachers: `old` (sweep-2 ensemble, reference), `big` (best <= 8-model ensemble of all
  non-distilled models from sweeps 1-3), `in2` (2 full-data models, in-sample), `oof` (same 2 configs, out-of-fold),
  plus a quick quality check of each teacher on training vs held-out sequences.
* **Phase 3:** 16 students: one per teacher (+ seed repeats), then recipe variations on `big`: 10 passes, width
  96/160/192, 3 layers, teacher-loss weighting, lower learning rate, dropout, 10% real-target loss.
* **Phase 4:** ensembles under cost budgets, speed on this CPU, upload-ready zips in Drive.

Needs sweeps 1-3 in Drive. Seed noise: ~0.0007 on validation (scored), ~0.005 on the holdout.

**Runtime:** Runtime > Change runtime type > **A100 or L4 GPU** and **High-RAM**.
Results go to Google Drive (`USE_DRIVE = True`), so they survive a disconnect.

Code below is generated from `pipeline/` by `colab/build_notebook.py`; edit there, not here.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!echo "CPUs: $(nproc)"; lscpu | grep "Model name"; free -g | head -2; df -h /content | tail -1

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
CPUs: 12
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:             167           2         162           0           3         164
overlay         236G   43G  194G  19% /


In [2]:
USE_DRIVE = True
OUT_DIR = '/content/drive/MyDrive/wunder-connectome/runs' if USE_DRIVE else '/content/runs'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
import os; os.makedirs(OUT_DIR, exist_ok=True); print('results ->', OUT_DIR)

Mounted at /content/drive
results -> /content/drive/MyDrive/wunder-connectome/runs


## 1. Data (33.7 GB; ~10-20 min)
Downloads to a resumable file, unpacks, then checks every parquet file opens with the right number of sequences. Skipped when the data is already there and valid; re-downloaded when anything is broken.

In [3]:
%%bash
mkdir -p /content/wnn && cd /content/wnn
check() {
python3 - <<'EOF'
import sys, pyarrow.parquet as pq
ok = True
for f, n in {'train': 10607, 'valid': 1873, 'valid_mask': 1873}.items():
    try:
        g = pq.ParquetFile(f'datasets/{f}.parquet').metadata.num_row_groups
        print(f'{f}: {g} sequences', 'OK' if g == n else f'(expected {n})'); ok &= g == n
    except Exception as e:
        print(f'{f}: BROKEN ({type(e).__name__}: {e})'); ok = False
sys.exit(0 if ok else 1)
EOF
}
if check; then echo 'data OK, nothing to do'; else
  rm -rf datasets
  URL=https://files.wundernn.io/wnn_connectome_starterpack.tar.gz
  for i in $(seq 10); do   # -C - resumes where a dropped connection stopped
    curl -L -C - --retry 20 --retry-all-errors -sS -o pack.tar.gz "$URL" && break
    echo "download interrupted, resuming ($i)"; sleep 5
  done
  ls -la pack.tar.gz
  tar -xzf pack.tar.gz --strip-components=1 --warning=no-unknown-keyword && find . -name '._*' -delete
  if check; then rm pack.tar.gz; echo 'data OK'; else echo 'DATA STILL BROKEN: delete /content/wnn/pack.tar.gz and re-run this cell'; exit 1; fi
fi
du -sh datasets

train: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/train.parquet'. Detail: [errno 2] No such file or directory)
valid: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid.parquet'. Detail: [errno 2] No such file or directory)
valid_mask: BROKEN (FileNotFoundError: [Errno 2] Failed to open local file 'datasets/valid_mask.parquet'. Detail: [errno 2] No such file or directory)
-rw-r--r-- 1 root root 33737976954 Oct 23 21:36 pack.tar.gz
train: 10607 sequences OK
valid: 1873 sequences OK
valid_mask: 1873 sequences OK
data OK
33G	datasets


In [4]:
!pip -q install onnx onnxruntime
%cd /content/wnn

/content/wnn


## 2. Pipeline code

In [5]:
!mkdir -p /content/wnn/pipeline

In [6]:
%%writefile /content/wnn/pipeline/__init__.py

Writing /content/wnn/pipeline/__init__.py


In [7]:
%%writefile /content/wnn/pipeline/data.py
"""Fast sequence loading from the competition parquet files.

Each row group is one 20k-step sequence. Readers keep one ParquetFile per thread
(re-opening parses a large footer) and write straight into a preallocated batch array.
"""
import threading
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pyarrow.parquet as pq

from utils import FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS, WARMUP

N_FEATURES = len(FEATURE_COLUMNS)


class SeqStore:
    def __init__(self, path: str, n_threads: int = 4):
        self.path = path
        pf = pq.ParquetFile(path)
        self.n_seqs = pf.num_row_groups
        self.has_mask = "is_scored" in pf.schema_arrow.names
        self.cols = list(FEATURE_COLUMNS) + list(TARGET_COLUMNS) + (["is_scored"] if self.has_mask else [])
        self._local = threading.local()
        self._pool = ThreadPoolExecutor(n_threads)
        self._cache = None

    def enable_cache(self):
        """Keep every sequence in RAM after its first read (~9 MB each)."""
        self._cache = {}

    def _pf(self):
        if not hasattr(self._local, "pf"):
            self._local.pf = pq.ParquetFile(self.path)
        return self._local.pf

    def _load_into(self, i, g, X, Y, M):
        if self._cache is not None and g in self._cache:
            X[i], Y[i], M[i] = self._cache[g]
            return
        df = self._pf().read_row_group(int(g), columns=self.cols, use_threads=False).to_pandas()
        a = df.to_numpy(np.float32)
        X[i] = a[:, :N_FEATURES]
        Y[i] = a[:, N_FEATURES:N_FEATURES + 2]
        need = np.arange(SEQUENCE_LENGTH) >= WARMUP
        M[i] = need & (a[:, -1] > 0.5) if self.has_mask else need
        if self._cache is not None:
            self._cache[g] = (X[i].copy(), Y[i].copy(), M[i].copy())

    def load(self, ids):
        """-> X [B,T,112] f32, Y [B,T,2] f32, M [B,T] bool (scored rows; all required rows if no mask)."""
        B = len(ids)
        X = np.empty((B, SEQUENCE_LENGTH, N_FEATURES), np.float32)
        Y = np.empty((B, SEQUENCE_LENGTH, 2), np.float32)
        M = np.empty((B, SEQUENCE_LENGTH), bool)
        list(self._pool.map(lambda ig: self._load_into(ig[0], ig[1], X, Y, M), enumerate(ids)))
        return X, Y, M

    def batches(self, ids, batch_size):
        """Yield loaded batches, prefetching the next one in the background."""
        chunks = [ids[i:i + batch_size] for i in range(0, len(ids), batch_size)]
        if not chunks:
            return
        prefetch = ThreadPoolExecutor(1)
        fut = prefetch.submit(self.load, chunks[0])
        for k in range(len(chunks)):
            batch = fut.result()
            if k + 1 < len(chunks):
                fut = prefetch.submit(self.load, chunks[k + 1])
            yield batch
        prefetch.shutdown()


def split_train(n_seqs: int, n_holdout: int, seed: int = 0):
    """Deterministic split of train row groups into (train_ids, holdout_ids)."""
    perm = np.random.default_rng(seed).permutation(n_seqs)
    return np.sort(perm[n_holdout:]), np.sort(perm[:n_holdout])

Writing /content/wnn/pipeline/data.py


In [8]:
%%writefile /content/wnn/pipeline/model.py
"""Stateful GRU/LSTM model, metric-aligned losses, ONNX step export and baseline import."""
import numpy as np
import torch
from torch import nn

N_FEATURES = 112
CLIP = 2.0


class GRUNet(nn.Module):
    """Stacked GRU (or LSTM with cell="lstm") + linear head. The recurrent module keeps the
    attribute name `gru` for either cell so older checkpoints load unchanged."""

    def __init__(self, hidden=128, layers=2, use_diff=False, dropout=0.0, cell="gru"):
        super().__init__()
        self.config = dict(hidden=hidden, layers=layers, use_diff=use_diff, dropout=dropout, cell=cell)
        self.use_diff = use_diff
        self.cell = cell
        in_dim = N_FEATURES * (2 if use_diff else 1)
        rnn = {"gru": nn.GRU, "lstm": nn.LSTM}[cell]
        self.gru = rnn(in_dim, hidden, layers, batch_first=True, dropout=dropout if layers > 1 else 0.0)
        self.head = nn.Linear(hidden, 2)

    def forward(self, x, h=None, x_prev=None):
        """x [B,T,112]; h [L,B,H] or None; x_prev [B,112] = last row of the previous chunk
        (None at sequence start, where the first diff is zero)."""
        if self.use_diff:
            first = x[:, :1] if x_prev is None else x_prev[:, None]
            prev = torch.cat([first, x[:, :-1]], 1)
            x = torch.cat([x, x - prev], -1)
        out, h = self.gru(x, h)
        return self.head(out), h


def detach_state(h):
    """Detach a recurrent state: tensor (GRU), (h, c) tuple (LSTM) or list of those (ensemble)."""
    if isinstance(h, torch.Tensor):
        return h.detach()
    return type(h)(detach_state(x) for x in h)


def macs(config) -> int:
    """Multiply-adds per row (inference cost model). Ensembles: sum over members."""
    if "members" in config:
        return sum(macs(m) for m in config["members"])
    gates = 4 if config.get("cell", "gru") == "lstm" else 3
    H, n_in = config["hidden"], N_FEATURES * (2 if config["use_diff"] else 1)
    total = 0
    for _ in range(config["layers"]):
        total += gates * H * (n_in + H)
        n_in = H
    return total + 2 * H


BASELINE_MACS = macs(dict(hidden=128, layers=2, use_diff=False))  # 190,720


def wmse_loss(p, y):
    """Weighted MSE with the metric's clipping and |y| weights. p, y: [N, 2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    return ((w * (p - y) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()


def distill_loss(p, teacher, y, weight="y"):
    """MSE to the teacher's predictions. Row weights: "y" = |clipped target| (like the metric),
    "none" = uniform, "teacher" = |teacher prediction| (no dependence on the noisy target)."""
    if weight == "y":
        w = y.clamp(-CLIP, CLIP).abs()
    elif weight == "teacher":
        w = teacher.abs()
    elif weight == "none":
        w = torch.ones_like(teacher)
    else:
        raise ValueError(weight)
    return ((w * (p - teacher) ** 2).sum(0) / (w.sum(0) + 1e-8)).mean()


def wpearson(p, y):
    """Per-target weighted Pearson (pooled over the rows given), metric conventions. -> [2]."""
    y = y.clamp(-CLIP, CLIP)
    w = y.abs()
    W = w.sum(0) + 1e-8
    yc = y - (w * y).sum(0) / W
    pc = p - (w * p).sum(0) / W
    cov = (w * yc * pc).sum(0)
    return cov / torch.sqrt((w * yc * yc).sum(0) * (w * pc * pc).sum(0) + 1e-12)


def compute_loss(kind, p, y, alpha=0.1):
    if kind == "wmse":
        return wmse_loss(p, y)
    if kind == "pearson":
        return 1 - wpearson(p, y).mean()
    if kind == "mix":  # correlation objective, small wmse term anchors the scale for clipping
        return 1 - wpearson(p, y).mean() + alpha * wmse_loss(p, y)
    raise ValueError(kind)


class EnsembleNet(nn.Module):
    """Average of several GRUNets. Each member's output is multiplied by `scales[k]` (per target)
    first, so members with different prediction spreads count equally. h is a list, one per member."""

    def __init__(self, nets, scales):
        super().__init__()
        self.nets = nn.ModuleList(nets)
        self.register_buffer("scales", torch.as_tensor(np.asarray(scales), dtype=torch.float32))  # [K, 2]
        self.use_diff = any(n.use_diff for n in nets)
        self.config = {"members": [n.config for n in nets], "scales": np.asarray(scales).tolist()}

    def forward(self, x, h=None, x_prev=None):
        h = h if h is not None else [None] * len(self.nets)
        preds, new_h = [], []
        for k, (net, hk) in enumerate(zip(self.nets, h)):
            p, hn = net(x, hk, x_prev if net.use_diff else None)
            preds.append(p * self.scales[k])
            new_h.append(hn)
        return torch.stack(preds).mean(0), new_h


def _split_layers(net):
    """Single-layer copies of a stacked GRU/LSTM, so the ONNX graph gets one state input per layer
    (as in the baseline) instead of slicing/concatenating a stacked state on every row."""
    rnn, mods = net.gru, []
    for l in range(rnn.num_layers):
        m = type(rnn)(rnn.input_size if l == 0 else rnn.hidden_size, rnn.hidden_size, 1, batch_first=True)
        for n in ("weight_ih", "weight_hh", "bias_ih", "bias_hh"):
            getattr(m, f"{n}_l0").data.copy_(getattr(rnn, f"{n}_l{l}").data)
        mods.append(m)
    return nn.ModuleList(mods)


class _Step(nn.Module):
    """One-row step graph: (x [1,1,112], x_prev [1,112], *states [1,1,H]) -> (pred [1,1,2], *new states).
    States are flat, per member and layer: GRU one h per layer, LSTM h and c per layer."""

    def __init__(self, net):
        super().__init__()
        self.ensemble = isinstance(net, EnsembleNet)
        members = list(net.nets) if self.ensemble else [net]
        self.layers = nn.ModuleList([_split_layers(m) for m in members])
        self.heads = nn.ModuleList([m.head for m in members])
        self.diff = [m.use_diff for m in members]
        self.lstm = [m.cell == "lstm" for m in members]
        self.register_buffer("scales", net.scales if self.ensemble else torch.ones(1, 2))

    def forward(self, x, x_prev, *states):
        states, new, preds, i = list(states), [], [], 0
        for k, layers in enumerate(self.layers):
            z = torch.cat([x, x - x_prev[:, None]], -1) if self.diff[k] else x
            for layer in layers:
                if self.lstm[k]:
                    z, (h, c) = layer(z, (states[i], states[i + 1]))
                    new += [h, c]
                    i += 2
                else:
                    z, h = layer(z, states[i])
                    new.append(h)
                    i += 1
            preds.append(self.heads[k](z) * self.scales[k] if self.ensemble else self.heads[k](z))
        pred = preds[0] if len(preds) == 1 else torch.stack(preds).mean(0)
        return (pred, *new)


def state_names(net):
    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]
    names = []
    for k, m in enumerate(members):
        for l in range(m.config["layers"]):
            names += [f"m{k}_h{l}"] + ([f"m{k}_c{l}"] if m.cell == "lstm" else [])
    return names


def export_onnx(net, path: str):
    """Export a GRUNet or EnsembleNet as a one-row step graph. Inputs: x, x_prev (dropped from the
    graph when no member uses diff features), then one state per member/layer named m{k}_h{l}
    (and m{k}_c{l} for LSTM); outputs: pred, then the new states in the same order."""
    net = net.eval().cpu()
    names = state_names(net)
    members = list(net.nets) if isinstance(net, EnsembleNet) else [net]
    states = [torch.zeros(1, 1, m.config["hidden"]) for m in members
              for _ in range(m.config["layers"] * (2 if m.cell == "lstm" else 1))]
    args = (torch.zeros(1, 1, N_FEATURES), torch.zeros(1, N_FEATURES), *states)
    with torch.no_grad():
        torch.onnx.export(_Step(net), args, path, input_names=["x", "x_prev", *names],
                          output_names=["pred", *[n + "_out" for n in names]], opset_version=17, dynamo=False)


def load_baseline(onnx_path: str) -> GRUNet:
    """Load the organisers' 2-layer GRU ONNX weights into a GRUNet (for checks / warm starts)."""
    import onnx
    from onnx import numpy_helper

    g = onnx.load(onnx_path).graph
    init = {t.name: numpy_helper.to_array(t) for t in g.initializer}
    grus = [n for n in g.node if n.op_type == "GRU"]
    matmul = next(n for n in g.node if n.op_type == "MatMul")
    H = 128
    net = GRUNet(hidden=H, layers=len(grus))

    def reorder(a):  # ONNX gates z,r,h -> PyTorch r,z,n
        return np.concatenate([a[H:2 * H], a[:H], a[2 * H:]], 0)

    sd = {}
    for l, n in enumerate(grus):
        W, R, B = (init[n.input[i]][0] for i in (1, 2, 3))
        sd[f"gru.weight_ih_l{l}"] = reorder(W)
        sd[f"gru.weight_hh_l{l}"] = reorder(R)
        sd[f"gru.bias_ih_l{l}"] = reorder(B[:3 * H])
        sd[f"gru.bias_hh_l{l}"] = reorder(B[3 * H:])
    sd["head.weight"] = init[matmul.input[1]].T
    sd["head.bias"] = init["reg_head.bias"]
    net.load_state_dict({k: torch.from_numpy(np.array(v, dtype=np.float32)) for k, v in sd.items()})
    return net

Writing /content/wnn/pipeline/model.py


In [9]:
%%writefile /content/wnn/pipeline/fast_eval.py
"""Batched evaluation: runs whole sequences through the network at once (seconds instead of
the ~15 min row-by-row loop) and scores them with the official GlobalAccumulator.

    python -m pipeline.fast_eval --baseline                 # should reproduce 0.617052
    python -m pipeline.fast_eval --ckpt runs/NAME/best.pt --holdout 500
"""
import argparse
import json
import time

import numpy as np
import torch

from utils import SEQUENCE_LENGTH, WARMUP, GlobalAccumulator

from .data import SeqStore, split_train
from .model import GRUNet, load_baseline

NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def predict_batch(net, X, device, chunk=2000):
    B, T, _ = X.shape
    out = np.empty((B, T, 2), np.float32)
    h = xp = None
    for s in range(0, T, chunk):
        x = torch.from_numpy(X[:, s:s + chunk]).to(device)
        p, h = net(x, h, xp)
        xp = x[:, -1]
        out[:, s:s + chunk] = p.float().cpu().numpy()
    return out


def evaluate_many(nets, store: SeqStore, ids, device, batch_size=128):
    """Score several models in one pass over the data. Per model:
    {'scored': WP on scored rows, 'all': WP on all required rows, ...} (equal without a mask)."""
    for net in nets:
        net.eval()
    accs = [(GlobalAccumulator(), GlobalAccumulator()) for _ in nets]
    for X, Y, M in store.batches(list(ids), batch_size):
        for net, (acc, acc_all) in zip(nets, accs):
            P = predict_batch(net, X, device)
            for b in range(len(X)):
                acc.add(Y[b], P[b], M[b])
                if store.has_mask:
                    acc_all.add(Y[b], P[b], NEED)
    out = []
    for acc, acc_all in accs:
        r = acc.result()
        res = {"scored": r["weighted_pearson"], "t0": r["t0"], "t1": r["t1"], "n_seqs": r["blocks"]}
        res["all"] = acc_all.result()["weighted_pearson"] if store.has_mask else res["scored"]
        out.append(res)
    return out


def evaluate(net, store: SeqStore, ids, device, batch_size=128):
    return evaluate_many([net], store, ids, device, batch_size)[0]


def load_ckpt(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    net = GRUNet(**ck["config"])
    net.load_state_dict(ck["state_dict"])
    return net


def main():
    ap = argparse.ArgumentParser()
    g = ap.add_mutually_exclusive_group(required=True)
    g.add_argument("--baseline", action="store_true")
    g.add_argument("--ckpt")
    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")
    ap.add_argument("--holdout", type=int, default=0, help="also score the N-seq train holdout")
    ap.add_argument("--device", default="auto")
    args = ap.parse_args()

    device = pick_device(args.device)
    net = (load_baseline("baseline/baseline.onnx") if args.baseline else load_ckpt(args.ckpt)).to(device)
    out = {}
    t = time.time()
    valid = SeqStore("datasets/valid.parquet")
    out["valid"] = evaluate(net, valid, range(args.valid_seqs or valid.n_seqs), device)
    if args.holdout:
        train = SeqStore("datasets/train.parquet")
        _, hold = split_train(train.n_seqs, args.holdout)
        out["holdout"] = evaluate(net, train, hold, device)
    out["seconds"] = round(time.time() - t, 1)
    print(json.dumps(out, indent=1))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/fast_eval.py


In [10]:
%%writefile /content/wnn/pipeline/trainer.py
"""Shared training loop: one data stream feeds any number of experiments.

Every batch of whole sequences is read from parquet once and then used to train each model in
turn (truncated BPTT, hidden state carried between `chunk`-step pieces, loss on steps >= 99).
Evaluation is shared the same way: one pass over the eval data scores all models.

Optional per model: EMA of the weights (evaluated and saved instead of the raw weights),
distillation towards a teacher's predictions on the training set (see pipeline/teacher.py), its own
number of passes, and a data fold to leave out (out-of-fold teachers: sequences with
id % n_folds == fold are skipped by that model).
"""
import copy
import json
import math
import os
import shutil
import time
from dataclasses import asdict, dataclass, field

import numpy as np
import torch

from utils import WARMUP

from .data import SeqStore, split_train
from .fast_eval import evaluate_many
from .model import BASELINE_MACS, GRUNet, compute_loss, detach_state, distill_loss, export_onnx, load_baseline, macs

HERE = os.path.dirname(os.path.abspath(__file__))


@dataclass
class ExpConfig:
    """Per-model settings. Everything about the data stream is shared (see Common)."""
    name: str
    hidden: int = 128
    layers: int = 2
    diff: bool = False
    dropout: float = 0.0
    init_baseline: bool = False
    loss: str = "wmse"          # wmse | pearson | mix
    alpha: float = 0.1          # wmse weight in mix
    lr: float = 2e-3
    wd: float = 1e-4
    seed: int = 0
    cell: str = "gru"           # gru | lstm
    chunk: int | None = None    # TBPTT length for this model (default: Common.chunk)
    ema: float = 0.0            # >0: keep an EMA of the weights with this decay; eval/save the EMA
    distill: float = 0.0        # weight of the teacher loss (0 = off, 1 = teacher only); needs a teacher
    distill_weight: str = "y"   # teacher-loss row weights: y (|clipped target|, like the metric) | none | teacher
    teacher: str | None = None  # this model's teacher .npy (default: Common.teacher)
    epochs: int | None = None   # passes for this model (default: Common.epochs)
    fold: int | None = None     # leave out training sequences with id % Common.n_folds == fold


@dataclass
class Common:
    epochs: int = 4
    batch: int = 256            # sequences per batch
    chunk: int = 500            # TBPTT length
    clip_grad: float = 1.0
    train_seqs: int | None = None
    holdout: int = 500
    valid_seqs: int = 400
    evals_per_epoch: int = 2
    seed: int = 0               # data order + holdout split
    device: str = "auto"
    threads: int = 4            # parquet reader threads
    cache_eval: bool = False    # keep eval data in RAM (~9 MB per sequence)
    cache_train: bool = False   # keep training sequences in RAM after first read
    data_dir: str = "datasets"
    out_dir: str = "runs"
    teacher: str | None = None  # teacher predictions on train.parquet (.npy, [n_seqs, 20000, 2]) for distillation
    n_folds: int = 4            # for ExpConfig.fold
    extra: dict = field(default_factory=dict)


class Experiment:
    def __init__(self, cfg: ExpConfig, common: Common, device, n_batches):
        self.cfg = cfg
        self.chunk = cfg.chunk or common.chunk
        self.epochs = cfg.epochs or common.epochs
        self.teacher_path = (cfg.teacher or common.teacher) if cfg.distill > 0 else None
        assert cfg.distill == 0 or self.teacher_path, f"{cfg.name}: distill > 0 needs a teacher"
        total_steps = self.epochs * n_batches * math.ceil(20000 / self.chunk)
        torch.manual_seed(cfg.seed)
        if cfg.init_baseline:
            assert not cfg.diff and cfg.hidden == 128 and cfg.layers == 2, "baseline shape is fixed"
            self.net = load_baseline("baseline/baseline.onnx")
        else:
            self.net = GRUNet(cfg.hidden, cfg.layers, cfg.diff, cfg.dropout, cfg.cell)
        self.net.to(device)
        self.ema_net = copy.deepcopy(self.net) if cfg.ema > 0 else None
        self.n_updates = 0
        self.total_steps = total_steps
        self.opt = torch.optim.AdamW(self.net.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        self.sched = torch.optim.lr_scheduler.OneCycleLR(self.opt, max_lr=cfg.lr, total_steps=total_steps,
                                                         pct_start=max(0.05, 3 / total_steps))
        self.dir = os.path.join(common.out_dir, cfg.name)
        os.makedirs(self.dir, exist_ok=True)
        self.log_f = open(os.path.join(self.dir, "log.jsonl"), "a")
        self.best = -1.0
        self.params = sum(p.numel() for p in self.net.parameters())
        self.cost = macs(self.net.config) / BASELINE_MACS

    @property
    def eval_net(self):
        return self.ema_net if self.ema_net is not None else self.net

    @torch.no_grad()
    def update_ema(self):
        self.n_updates += 1
        if self.ema_net is None:
            return
        decay = min(self.cfg.ema, (1 + self.n_updates) / (10 + self.n_updates))  # warm-up
        for pe, p in zip(self.ema_net.parameters(), self.net.parameters()):
            pe.lerp_(p, 1 - decay)

    def log(self, **kw):
        line = json.dumps(kw)
        self.log_f.write(line + "\n")
        self.log_f.flush()
        print(f"[{self.cfg.name}] {line}", flush=True)


def _store(common, name):
    return SeqStore(os.path.join(common.data_dir, f"{name}.parquet"), n_threads=common.threads)


def train_many(configs: list[ExpConfig], common: Common):
    from .fast_eval import pick_device

    device = pick_device(common.device)
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    rng = np.random.default_rng(common.seed)
    train, valid = _store(common, "train"), _store(common, "valid")
    if common.cache_train:
        train.enable_cache()
    if common.cache_eval:
        valid.enable_cache()
    train_ids, hold_ids = split_train(train.n_seqs, common.holdout, common.seed)
    if common.train_seqs:
        train_ids = rng.choice(train_ids, common.train_seqs, replace=False)
    valid_ids = list(range(min(common.valid_seqs, valid.n_seqs)))
    hold_store = train
    if common.cache_eval and not common.cache_train:  # cache holdout without caching all of train
        hold_store = _store(common, "train")
        hold_store.enable_cache()

    n_batches = math.ceil(len(train_ids) / common.batch)
    exps = [Experiment(c, common, device, n_batches) for c in configs]
    teachers = {}  # path -> memmap, loaded once and shared by the models that use it
    for path in sorted({e.teacher_path for e in exps if e.teacher_path}):
        teachers[path] = np.load(path, mmap_mode="r")
        assert teachers[path].shape[:2] == (train.n_seqs, 20000), f"teacher {path}: shape {teachers[path].shape}"
    n_epochs = max(e.epochs for e in exps)
    t0 = time.time()
    for e in exps:
        e.log(event="start", config=asdict(e.cfg), common=asdict(common), device=str(device), params=e.params,
              cost=round(e.cost, 3), train_seqs=len(train_ids), holdout=len(hold_ids), steps=e.total_steps)
    eval_at = set(np.linspace(0, n_batches, common.evals_per_epoch + 1, dtype=int)[1:] - 1)

    def run_eval(epoch, bi, group):
        nets = [e.eval_net for e in group]
        vs = evaluate_many(nets, valid, valid_ids, device)
        hs = evaluate_many(nets, hold_store, hold_ids, device)
        for e, v, h in zip(group, vs, hs):
            sel = (v["scored"] + h["all"]) / 2  # don't select on the visible valid mask alone
            is_best = sel > e.best
            if is_best:
                e.best = sel
                torch.save({"config": e.net.config, "state_dict": e.eval_net.state_dict(), "exp": asdict(e.cfg),
                            "n_folds": common.n_folds},
                           os.path.join(e.dir, "best.pt"))
            e.log(event="eval", epoch=epoch, batch=bi, valid=round(v["scored"], 6), valid_all=round(v["all"], 6),
                  holdout=round(h["all"], 6), select=round(sel, 6), best=is_best, time=round(time.time() - t0, 1))
            e.net.train()

    for epoch in range(n_epochs):
        active = [e for e in exps if epoch < e.epochs]
        for e in active:
            e.net.train()
        ids = np.asarray(rng.permutation(train_ids))
        for bi, (X, Y, _) in enumerate(train.batches(list(ids), common.batch)):
            tb = time.time()
            batch_ids = ids[bi * common.batch:(bi + 1) * common.batch]
            Xd = torch.from_numpy(X).to(device, non_blocking=True)  # one host->device copy per batch
            Yd = torch.from_numpy(Y).to(device, non_blocking=True)
            Td = {path: torch.from_numpy(np.asarray(t[np.sort(batch_ids)], dtype=np.float32)).to(device)
                  [torch.as_tensor(np.argsort(np.argsort(batch_ids)), device=device)]  # memmap reads in file order
                  for path, t in teachers.items() if any(e.teacher_path == path for e in active)}
            for e in active:
                Xe, Ye, Te = Xd, Yd, Td.get(e.teacher_path)
                if e.cfg.fold is not None:  # out-of-fold model: drop its fold's sequences from the batch
                    rows = torch.as_tensor(np.flatnonzero(batch_ids % common.n_folds != e.cfg.fold), device=device)
                    Xe, Ye = Xd[rows], Yd[rows]
                    Te = Te[rows] if Te is not None else None
                h = xp = None
                losses = []
                for s in range(0, X.shape[1], e.chunk):
                    x, y = Xe[:, s:s + e.chunk], Ye[:, s:s + e.chunk]
                    p, h = e.net(x, h, xp)
                    lo = max(0, WARMUP - s)  # skip warm-up rows
                    if lo < x.shape[1]:
                        pf, yf = p[:, lo:].reshape(-1, 2), y[:, lo:].reshape(-1, 2)
                        lam = e.cfg.distill
                        loss = compute_loss(e.cfg.loss, pf, yf, e.cfg.alpha) if lam < 1 else 0.0
                        if lam > 0:
                            tf = Te[:, s:s + e.chunk][:, lo:].reshape(-1, 2)
                            loss = (1 - lam) * loss + lam * distill_loss(pf, tf, yf, e.cfg.distill_weight)
                        e.opt.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(e.net.parameters(), common.clip_grad)
                        e.opt.step()
                        e.sched.step()
                        e.update_ema()
                        losses.append(loss.item())
                    h = detach_state(h)
                    xp = x[:, -1]
                e.log(event="batch", epoch=epoch, batch=bi, loss=round(float(np.mean(losses)), 5),
                      lr=round(e.sched.get_last_lr()[0], 6))
            print(f"batch {bi + 1}/{n_batches} epoch {epoch + 1}/{n_epochs}: {time.time() - tb:.1f}s "
                  f"for {len(active)} models, elapsed {(time.time() - t0) / 60:.1f} min", flush=True)
            del Xd, Yd, Td
            if bi in eval_at:
                run_eval(epoch, bi, active)

    # final: best checkpoints -> full validation + holdout, ONNX submission folders, summary table
    for e in exps:
        ck = torch.load(os.path.join(e.dir, "best.pt"), map_location="cpu", weights_only=False)
        e.net.load_state_dict(ck["state_dict"])
        e.net.to(device)
    nets = [e.net for e in exps]
    full_valid = evaluate_many(nets, _store(common, "valid"), range(valid.n_seqs), device)  # uncached: 17 GB
    full_hold = evaluate_many(nets, hold_store, hold_ids, device)
    rows = []
    for e, v, h in zip(exps, full_valid, full_hold):
        sub = os.path.join(e.dir, "submission")
        os.makedirs(sub, exist_ok=True)
        export_onnx(e.net, os.path.join(sub, "model.onnx"))
        shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(sub, "solution.py"))
        res = {"valid_full": v, "holdout": h}
        json.dump(res, open(os.path.join(e.dir, "results.json"), "w"), indent=1)
        e.log(event="done", **res)
        rows.append({"name": e.cfg.name, "params": e.params, "cost": round(e.cost, 2), "valid_scored": round(v["scored"], 6),
                     "valid_all": round(v["all"], 6), "holdout_all": round(h["all"], 6),
                     "t0": round(v["t0"], 4), "t1": round(v["t1"], 4)})
    rows.sort(key=lambda r: -(r["valid_scored"] + r["holdout_all"]))
    with open(os.path.join(common.out_dir, "summary.jsonl"), "a") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("\n" + " | ".join(rows[0].keys()))
    for r in rows:
        print(" | ".join(str(v) for v in r.values()))
    return rows

Writing /content/wnn/pipeline/trainer.py


In [11]:
%%writefile /content/wnn/pipeline/multi_train.py
"""Train several experiments side by side from one data stream.

    python -m pipeline.multi_train --config experiments.json --epochs 4 --threads 8 --cache-eval

experiments.json is a list of ExpConfig dicts, e.g.
    [{"name": "wmse"}, {"name": "mix", "loss": "mix"}, {"name": "diff_h192", "diff": true, "hidden": 192}]
Shared settings (epochs, batch, chunk, data/out dirs, ...) are the Common fields, as flags.
"""
import argparse
import dataclasses
import json
import typing

from .trainer import Common, ExpConfig, train_many


def field_type(f):
    """int | None -> int, str | None -> str, plain types unchanged."""
    args = [a for a in typing.get_args(f.type) if a is not type(None)]
    return args[0] if args else f.type


def add_flags(ap, cls, skip=("extra",)):
    for f in dataclasses.fields(cls):
        if f.name in skip:
            continue
        flag = "--" + f.name.replace("_", "-")
        if field_type(f) is bool:
            ap.add_argument(flag, action="store_true")
        else:
            ap.add_argument(flag, type=field_type(f), default=f.default)


def add_common_flags(ap):
    add_flags(ap, Common)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True, help="JSON list of experiment configs")
    add_common_flags(ap)
    args = vars(ap.parse_args())
    configs = [ExpConfig(**c) for c in json.load(open(args.pop("config")))]
    names = [c.name for c in configs]
    assert len(set(names)) == len(names), "experiment names must be unique"
    train_many(configs, Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/multi_train.py


In [12]:
%%writefile /content/wnn/pipeline/train.py
"""Train one stateful GRU (thin wrapper over trainer.train_many; see multi_train.py for several).

    python -m pipeline.train --name quick --train-seqs 1000 --epochs 2
    python -m pipeline.train --name gru128_wmse --epochs 4

Outputs in runs/NAME/: best.pt, log.jsonl, results.json, submission/{solution.py, model.onnx}.
"""
import argparse
import dataclasses

from .multi_train import add_common_flags, add_flags
from .trainer import Common, ExpConfig, train_many


def main():
    ap = argparse.ArgumentParser()
    shared = ("seed", "chunk")  # one flag each for a single model: used for the model and the data stream
    exp_fields = {f.name for f in dataclasses.fields(ExpConfig)} - set(shared)
    ap.add_argument("--name", required=True)
    add_flags(ap, ExpConfig, skip=("name", *shared))
    add_common_flags(ap)
    args = vars(ap.parse_args())
    exp = ExpConfig(**{k: args.pop(k) for k in list(args) if k in exp_fields}, seed=args["seed"])
    train_many([exp], Common(**args))


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/train.py


In [13]:
%%writefile /content/wnn/pipeline/ensemble.py
"""Score models alone and averaged, pick an ensemble on the train holdout, optionally export it.

    python -m pipeline.ensemble --ckpts "runs/*/best.pt" --budgets 1 1.5 2 --export runs/ensembles

Every model is run once over the holdout and validation data (shared loading); combinations are then
scored from the stored predictions. Members are rescaled to a common prediction spread before
averaging (scales measured on the holdout). Selection uses the holdout only, so the validation
numbers stay an honest check of the chosen combination.

Cost = multiply-adds per row relative to the baseline (models.macs). Real run time also has a
fixed per-row overhead, so an ensemble's measured time ratio is lower than its cost; check
candidates with pipeline.speed. One greedy ensemble is built and exported per --budgets value.
"""
import argparse
import glob
import json
import os
import shutil
import time

import numpy as np

from utils import SEQUENCE_LENGTH, WARMUP, weighted_pearson

from .data import SeqStore, split_train
from .fast_eval import load_ckpt, pick_device, predict_batch
from .model import BASELINE_MACS, EnsembleNet, export_onnx, load_baseline, macs

HERE = os.path.dirname(os.path.abspath(__file__))
NEED = np.arange(SEQUENCE_LENGTH) >= WARMUP


def load_member(path):
    if path == "baseline":
        return "baseline", load_baseline("baseline/baseline.onnx")
    return os.path.basename(os.path.dirname(path)), load_ckpt(path)


def collect(nets, store, ids, device, masks):
    """-> Y {mask: [N,2]}, P {mask: [K,N,2]} for masks 'all' (required rows) and/or 'scored'."""
    ys = {m: [] for m in masks}
    ps = {m: [[] for _ in nets] for m in masks}
    for X, Y, M in store.batches(list(ids), 128):
        sel = {"all": np.broadcast_to(NEED, M.shape), "scored": M}
        for m in masks:
            ys[m].append(Y[sel[m]])
        for k, net in enumerate(nets):
            P = predict_batch(net, X, device)
            for m in masks:
                ps[m][k].append(P[sel[m]])
    return ({m: np.concatenate(ys[m]) for m in masks},
            {m: np.stack([np.concatenate(p) for p in ps[m]]) for m in masks})


def wp(y, p):
    return float(np.mean([weighted_pearson(y[:, j], p[:, j]) for j in range(2)]))


def combine(P, scales, idx):
    return (P[idx] * scales[idx][:, None, :]).mean(0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--ckpts", nargs="+", required=True, help="best.pt paths/globs, or 'baseline'")
    ap.add_argument("--holdout", type=int, default=500)
    ap.add_argument("--valid-seqs", type=int, default=None, help="first N validation seqs (default all)")
    ap.add_argument("--max-members", type=int, default=5)
    ap.add_argument("--budgets", type=float, nargs="+", default=[99.0],
                    help="max total cost (x baseline multiply-adds); one greedy ensemble per budget")
    ap.add_argument("--export", help="root folder: writes <root>/cost<budget>/submission per budget")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--threads", type=int, default=4)
    args = ap.parse_args()

    paths = []
    for c in args.ckpts:
        paths += [c] if c == "baseline" else sorted(glob.glob(c))
    names, nets = zip(*[load_member(p) for p in paths])
    device = pick_device(args.device)
    nets = [n.eval().to(device) for n in nets]
    print(f"{len(nets)} models: {', '.join(names)}", flush=True)

    t = time.time()
    train = SeqStore("datasets/train.parquet", args.threads)
    valid = SeqStore("datasets/valid.parquet", args.threads)
    _, hold_ids = split_train(train.n_seqs, args.holdout)
    Yh, Ph = collect(nets, train, hold_ids, device, ["all"])
    Yv, Pv = collect(nets, valid, range(args.valid_seqs or valid.n_seqs), device, ["scored", "all"])
    print(f"predictions collected in {time.time() - t:.0f}s", flush=True)

    sd = Ph["all"].std(axis=1)  # [K, 2]
    scales = sd.mean(0, keepdims=True) / sd

    def score(idx):
        return {"holdout": wp(Yh["all"], combine(Ph["all"], scales, idx)),
                "valid_scored": wp(Yv["scored"], combine(Pv["scored"], scales, idx)),
                "valid_all": wp(Yv["all"], combine(Pv["all"], scales, idx))}

    cost = np.array([macs(n.config) / BASELINE_MACS for n in nets])
    rows = []
    singles = [(k, score([k])) for k in range(len(nets))]
    for k, sc in singles:
        rows.append({"combo": names[k], "n": 1, "cost": cost[k], **sc})
    order = [k for k, sc in sorted(singles, key=lambda ks: -ks[1]["holdout"])]
    for top in range(2, min(len(nets), args.max_members) + 1):
        rows.append({"combo": f"top{top} by holdout", "n": top, "cost": cost[order[:top]].sum(),
                     "members": [names[k] for k in order[:top]], **score(order[:top])})

    def greedy(budget):
        """Forward selection on the holdout (no repeats) within a total cost budget."""
        ok = [k for k in order if cost[k] <= budget]
        if not ok:
            return []
        chosen, best = [ok[0]], singles[ok[0]][1]["holdout"]
        while len(chosen) < args.max_members:
            cand = [(k, wp(Yh["all"], combine(Ph["all"], scales, chosen + [k]))) for k in range(len(nets))
                    if k not in chosen and cost[chosen + [k]].sum() <= budget]
            if not cand:
                break
            k, sc = max(cand, key=lambda c: c[1])
            if sc <= best:
                break
            chosen.append(k)
            best = sc
        return chosen

    picks = {}
    for b in args.budgets:
        chosen = greedy(b)
        if chosen:
            picks[b] = chosen
            rows.append({"combo": f"greedy, cost <= {b:g}", "n": len(chosen), "cost": cost[chosen].sum(),
                         "members": [names[k] for k in chosen], **score(chosen)})

    print(f"\n{'combo':30s} {'n':>2s} {'cost':>5s} {'holdout':>8s} {'valid_sc':>8s} {'valid_all':>9s}")
    for r in rows:
        print(f"{r['combo'][:30]:30s} {r['n']:2d} {r['cost']:5.2f} {r['holdout']:8.5f} {r['valid_scored']:8.5f} "
              f"{r['valid_all']:9.5f}" + (f"   {'+'.join(r['members'])}" if "members" in r else ""))

    if args.export:
        for b, chosen in picks.items():
            out = os.path.join(args.export, f"cost{b:g}", "submission")
            os.makedirs(out, exist_ok=True)
            net = nets[chosen[0]] if len(chosen) == 1 else EnsembleNet([nets[k] for k in chosen], scales[chosen])
            export_onnx(net, os.path.join(out, "model.onnx"))
            shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(out, "solution.py"))
            row = next(r for r in rows if r["combo"] == f"greedy, cost <= {b:g}")
            json.dump({"members": [paths[k] for k in chosen], "scales": scales[chosen].tolist(),
                       "scores": {k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in row.items()}},
                      open(os.path.join(out, "ensemble.json"), "w"), indent=1)
            print(f"exported cost<={b:g}: {len(chosen)} model(s) -> {out}")

if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/ensemble.py


In [14]:
%%writefile /content/wnn/pipeline/speed.py
"""Time submission folders row by row on one CPU thread, next to the official baseline.

    python -m pipeline.speed runs/NAME/submission runs/ens/submission

The test set is 39.4M rows in 60 min (~91 us/row on the grader's CPU). The baseline is known to
fit, so the ratio to the baseline is the number to watch; absolute times depend on this machine.
"""
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"

import argparse
import importlib.util
import sys
import time

import numpy as np
import pyarrow.parquet as pq

from utils import FEATURE_COLUMNS, WARMUP, DataPoint

TEST_ROWS = 39_400_000


def load_model(folder, i):
    folder = os.path.abspath(folder)
    sys.path.insert(0, folder)
    spec = importlib.util.spec_from_file_location(f"solution_{i}", os.path.join(folder, "solution.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.PredictionModel()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("folders", nargs="+")
    ap.add_argument("--rows", type=int, default=20000)
    args = ap.parse_args()

    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, columns=list(FEATURE_COLUMNS))
    X = np.column_stack([tbl[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)[:args.rows]

    results = []
    for i, folder in enumerate(["baseline"] + args.folders):
        model = load_model(folder, i)
        t = time.perf_counter()
        for step in range(len(X)):
            model.predict(DataPoint(0, step, step >= WARMUP, X[step]))
        us = (time.perf_counter() - t) / len(X) * 1e6
        results.append((folder, us))
        base = results[0][1]
        print(f"{folder:50s} {us:7.1f} us/row  {us / base:5.2f}x baseline  "
              f"~{us * TEST_ROWS / 60e6:5.1f} min for the test set here", flush=True)


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/speed.py


In [15]:
%%writefile /content/wnn/pipeline/teacher.py
"""Write a teacher's predictions on every training sequence, as the target for distillation.

    python -m pipeline.teacher --ensemble runs/ens/cost99/submission/ensemble.json --out teacher.npy
    python -m pipeline.teacher --ckpts runs/sweep2/s1_wmse_h128/best.pt runs/sweep2/c3_*/best.pt --out t_in2.npy
    python -m pipeline.teacher --ckpts "runs/sweep4/f*/best.pt" --oof --out t_oof.npy

Output: float16 .npy of shape [n_train_seqs, 20000, 2] (~850 MB), indexed by train.parquet row group,
read by the trainer (Common.teacher / ExpConfig.teacher).

In-sample (default): every member predicts every sequence, including the ones it trained on.
--oof: members were trained with ExpConfig.fold; sequence i is predicted only by the members whose
left-out fold is i % n_folds, so every training sequence gets honest (out-of-fold) predictions.
With --ckpts, members are rescaled to a common prediction spread (measured on the first batch).
"""
import argparse
import glob
import json
import time

import numpy as np
import torch

from .data import SeqStore
from .fast_eval import load_ckpt, pick_device, predict_batch
from .model import EnsembleNet


def main():
    ap = argparse.ArgumentParser()
    src = ap.add_mutually_exclusive_group(required=True)
    src.add_argument("--ensemble", help="ensemble.json written by pipeline.ensemble --export")
    src.add_argument("--ckpts", nargs="+", help="best.pt paths/globs, averaged")
    ap.add_argument("--oof", action="store_true", help="out-of-fold: use each member only on its left-out fold")
    ap.add_argument("--out", required=True)
    ap.add_argument("--data", default="datasets/train.parquet")
    ap.add_argument("--limit", type=int, default=None, help="only predict the first N sequences (tests)")
    ap.add_argument("--threads", type=int, default=8)
    ap.add_argument("--device", default="auto")
    args = ap.parse_args()
    device = pick_device(args.device)

    if args.ensemble:
        spec = json.load(open(args.ensemble))
        paths, scales = spec["members"], np.asarray(spec["scales"])
    else:
        paths = [p for c in args.ckpts for p in sorted(glob.glob(c))]
        scales = None
    cks = [torch.load(p, map_location="cpu", weights_only=False) for p in paths]
    nets = [load_ckpt(p).eval().to(device) for p in paths]
    folds = [ck.get("exp", {}).get("fold") for ck in cks]
    if args.oof:
        assert all(f is not None for f in folds), "--oof needs members trained with a fold"
        n_folds = cks[0].get("n_folds") or max(folds) + 1
    print(f"{len(nets)} members" + (f", folds {sorted(set(folds))}" if args.oof else ""), flush=True)

    store = SeqStore(args.data, args.threads)
    n = store.n_seqs
    out = np.lib.format.open_memmap(args.out, mode="w+", dtype=np.float16, shape=(n, 20000, 2))
    ids = list(range(args.limit or n))
    if args.limit:
        out[args.limit:] = 0
    t, done = time.time(), 0
    for X, _, _ in store.batches(ids, 128):
        P = np.stack([predict_batch(net, X, device) for net in nets])  # [K, B, T, 2]
        if scales is None:  # common spread, fixed from the first batch
            sd = P[:, :, 99:].std(axis=(1, 2))  # [K, 2]
            scales = sd.mean(0, keepdims=True) / sd
        P = P * scales[:, None, None, :]
        if args.oof:
            batch_fold = np.arange(done, done + len(X)) % n_folds
            pred = np.zeros(P.shape[1:], np.float32)
            for f in range(n_folds):
                members = [k for k in range(len(nets)) if folds[k] == f]
                rows = batch_fold == f
                pred[rows] = P[members][:, rows].mean(0)
        else:
            pred = P.mean(0)
        out[done:done + len(X)] = pred.astype(np.float16)
        done += len(X)
        if done % 1280 == 0 or done == len(ids):
            print(f"{done}/{len(ids)} sequences, {time.time() - t:.0f}s", flush=True)
    out.flush()
    print(f"teacher ({len(nets)} models{', out-of-fold' if args.oof else ''}) -> {args.out}")


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/teacher.py


In [16]:
%%writefile /content/wnn/pipeline/test_export.py
"""Check that exported ONNX step graphs + solution.py reproduce PyTorch predictions row by row.

    python -m pipeline.test_export            # random GRU / diff-GRU / LSTM / mixed ensemble
"""
import importlib.util
import os
import shutil
import tempfile

import numpy as np
import pyarrow.parquet as pq
import torch

from utils import FEATURE_COLUMNS, WARMUP, DataPoint

from .fast_eval import predict_batch
from .model import EnsembleNet, GRUNet, export_onnx

HERE = os.path.dirname(os.path.abspath(__file__))


def run_rows(net, X):
    d = tempfile.mkdtemp()
    export_onnx(net, os.path.join(d, "model.onnx"))
    shutil.copy(os.path.join(HERE, "solution_template.py"), os.path.join(d, "solution.py"))
    spec = importlib.util.spec_from_file_location("sol_test", os.path.join(d, "solution.py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    m = mod.PredictionModel()
    out = np.zeros((len(X), 2), np.float32)
    for seq in (0, 1):  # two sequences: also checks the state reset
        for t in range(len(X)):
            p = m.predict(DataPoint(seq, t, t >= WARMUP, X[t]))
            if p is not None:
                out[t] = p
    shutil.rmtree(d)
    return out


def main():
    torch.manual_seed(0)
    tbl = pq.ParquetFile("datasets/valid.parquet").read_row_group(0, columns=list(FEATURE_COLUMNS))
    X = np.column_stack([tbl[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)[:3000]
    a, b, c = GRUNet(96), GRUNet(64, use_diff=True), GRUNet(80, layers=3, cell="lstm")
    nets = {"gru": a, "gru_diff": b, "lstm_3layer": c,
            "ensemble": EnsembleNet([a, b, c], np.array([[1.0, 1.1], [0.9, 1.2], [1.3, 0.7]]))}
    for name, net in nets.items():
        net.eval()
        ref = predict_batch(net, X[None], torch.device("cpu"), chunk=700)[0]
        got = run_rows(net, X)
        err = np.abs(ref[WARMUP:] - got[WARMUP:]).max()
        print(f"{name:12s} max |torch - onnx| = {err:.2e}  {'OK' if err < 1e-4 else 'MISMATCH'}")


if __name__ == "__main__":
    main()

Writing /content/wnn/pipeline/test_export.py


In [17]:
%%writefile /content/wnn/pipeline/solution_template.py
"""Generic stateful ONNX step runner. Copied as solution.py next to model.onnx by the pipeline.
Works for single models and ensembles: every graph input other than x / x_prev is a recurrent
state, and graph outputs[1:] are their updates in the same order."""
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import onnxruntime as ort


class PredictionModel:
    def __init__(self):
        o = ort.SessionOptions()
        o.intra_op_num_threads = 1
        o.inter_op_num_threads = 1
        o.execution_mode = ort.ExecutionMode.ORT_SEQUENTIAL
        o.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
        o.add_session_config_entry("session.intra_op.allow_spinning", "0")
        self.session = ort.InferenceSession(str(Path(__file__).resolve().parent / "model.onnx"),
                                            sess_options=o, providers=["CPUExecutionProvider"])
        inputs = self.session.get_inputs()
        self.state_names = [i.name for i in inputs if i.name not in ("x", "x_prev")]
        self.state_init = [np.zeros(i.shape, dtype=np.float32) for i in inputs if i.name in self.state_names]
        self.use_prev = any(i.name == "x_prev" for i in inputs)  # absent when no model uses diff features
        self.out_names = [o.name for o in self.session.get_outputs()]
        self.seq_ix = None

    def _reset(self):
        self.feeds = dict(zip(self.state_names, [s.copy() for s in self.state_init]))
        self.x_prev = None

    def predict(self, data_point):
        if data_point.seq_ix != self.seq_ix:
            self.seq_ix = data_point.seq_ix
            self._reset()
        feeds = self.feeds
        x = np.asarray(data_point.state, dtype=np.float32).reshape(1, 1, 112)
        feeds["x"] = x
        if self.use_prev:
            feeds["x_prev"] = x[0] if self.x_prev is None else self.x_prev
            self.x_prev = x[0]
        out = self.session.run(self.out_names, feeds)
        for name, value in zip(self.state_names, out[1:]):
            feeds[name] = value
        pred = out[0].reshape(2)
        if not np.isfinite(pred).all():  # never seen; keep the submission valid and restart the state
            self._reset()
            pred = np.zeros(2, dtype=np.float32)
        if not data_point.need_prediction:
            return None
        return pred

Writing /content/wnn/pipeline/solution_template.py


## 3. Sanity check: baseline weights through our fast evaluator (300 seqs: 0.6246; full valid: 0.617052)

In [18]:
!python -m pipeline.fast_eval --baseline --valid-seqs 300

{
 "valid": {
  "scored": 0.6246286498889674,
  "t0": 0.6038379686442771,
  "t1": 0.6454193311336577,
  "n_seqs": 300,
  "all": 0.4380707773677913
 },
 "seconds": 4.8
}


## 4. Phase 1: out-of-fold teacher models

In [19]:
import json
experiments = [{'name': 'f0_wmse_h128', 'fold': 0},
 {'name': 'f1_wmse_h128', 'fold': 1},
 {'name': 'f2_wmse_h128', 'fold': 2},
 {'name': 'f3_wmse_h128', 'fold': 3},
 {'name': 'f0_mixdiff_h192', 'fold': 0, 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1},
 {'name': 'f1_mixdiff_h192', 'fold': 1, 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1},
 {'name': 'f2_mixdiff_h192', 'fold': 2, 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1},
 {'name': 'f3_mixdiff_h192', 'fold': 3, 'loss': 'mix', 'diff': True, 'hidden': 192, 'dropout': 0.1}]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')

8 experiments


In [20]:
TEACH = f'{OUT_DIR}/sweep4_teachers'
!python -m pipeline.multi_train --config experiments.json --epochs 5 --evals-per-epoch 1 --threads 8 --cache-eval --out-dir {TEACH} 2>&1 | grep -v '"event": "batch"'

[f0_wmse_h128] {"event": "start", "config": {"name": "f0_wmse_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 0, "cell": "gru", "chunk": null, "ema": 0.0, "distill": 0.0, "distill_weight": "y", "teacher": null, "epochs": null, "fold": 0}, "common": {"epochs": 5, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_per_epoch": 1, "seed": 0, "device": "auto", "threads": 8, "cache_eval": true, "cache_train": false, "data_dir": "datasets", "out_dir": "/content/drive/MyDrive/wunder-connectome/runs/sweep4_teachers", "teacher": null, "n_folds": 4, "extra": {}}, "device": "cuda", "params": 192258, "cost": 1.0, "train_seqs": 10107, "holdout": 500, "steps": 8000}
[f1_wmse_h128] {"event": "start", "config": {"name": "f1_wmse_h128", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha":

## 5. Phase 2: teachers

In [21]:
# old: the sweep-2 ensemble (sweep 3's teacher)
!python -m pipeline.teacher --ensemble {OUT_DIR}/ensemble_sweep2/submission/ensemble.json --out /content/wnn/t_old.npy
# in2 / oof: the same two configs, trained on all data (sweep 2) vs out-of-fold (phase 1)
!python -m pipeline.teacher --ckpts {OUT_DIR}/sweep2/s1_wmse_h128/best.pt {OUT_DIR}/sweep2/c3_mix_diff_h192_drop/best.pt --out /content/wnn/t_in2.npy
!python -m pipeline.teacher --ckpts "{TEACH}/f*/best.pt" --oof --out /content/wnn/t_oof.npy

4 members
1280/10607 sequences, 28s
2560/10607 sequences, 53s
3840/10607 sequences, 78s
5120/10607 sequences, 103s
6400/10607 sequences, 128s
7680/10607 sequences, 153s
8960/10607 sequences, 178s
10240/10607 sequences, 203s
10607/10607 sequences, 210s
teacher (4 models) -> /content/wnn/t_old.npy
2 members
1280/10607 sequences, 26s
2560/10607 sequences, 49s
3840/10607 sequences, 72s
5120/10607 sequences, 95s
6400/10607 sequences, 119s
7680/10607 sequences, 142s
8960/10607 sequences, 165s
10240/10607 sequences, 188s
10607/10607 sequences, 195s
teacher (2 models) -> /content/wnn/t_in2.npy
8 members, folds [0, 1, 2, 3]
1280/10607 sequences, 89s
2560/10607 sequences, 175s
3840/10607 sequences, 260s
5120/10607 sequences, 345s
6400/10607 sequences, 431s
7680/10607 sequences, 516s
8960/10607 sequences, 601s
10240/10607 sequences, 687s
10607/10607 sequences, 711s
teacher (8 models, out-of-fold) -> /content/wnn/t_oof.npy


In [22]:
# big: greedy ensemble (no cost limit, up to 8 members) over every non-distilled model so far
!python -m pipeline.ensemble --ckpts "{OUT_DIR}/sweep3/[!d]*/best.pt" "{OUT_DIR}/sweep2/*/best.pt" "{OUT_DIR}/e0[12456]*/best.pt" --threads 8 --max-members 8 --budgets 99 --export {OUT_DIR}/teacher_big 2>&1 | grep -vE 'Warning|_generic_rnn|warnings.warn'
!python -m pipeline.teacher --ensemble {OUT_DIR}/teacher_big/cost99/submission/ensemble.json --out /content/wnn/t_big.npy

24 models: chunk2000_h128, ema_h128, h64, h64_s1, h96, h96_s1, l1_h192, l3_h96, lr4e-3_h128, lstm_h112, r0_wmse_h128, c1_mix_diff_h128, c2_mix_diff_h128_drop, c3_mix_diff_h192_drop, c3b_mix_diff_h192_drop_s1, c4_mix_diff_h256_drop, s0_wmse_h128, s1_wmse_h128, s2_wmse_h128, e01_wmse_h128, e02_mix_h128, e04_wmse_diff_h128, e05_wmse_h192_drop, e06_wmse_h128_lr1e-3
predictions collected in 371s

combo                           n  cost  holdout valid_sc valid_all
chunk2000_h128                  1  1.00  0.44721  0.67598   0.50403
ema_h128                        1  1.00  0.44982  0.67733   0.50602
h64                             1  0.31  0.44755  0.67624   0.50364
h64_s1                          1  0.31  0.44716  0.67512   0.50409
h96                             1  0.61  0.44935  0.67752   0.50579
h96_s1                          1  0.61  0.44792  0.67773   0.50606
l1_h192                         1  0.92  0.44726  0.67558   0.50321
l3_h96                          1  0.89  0.45004  0.67890   0

Teacher quality: weighted Pearson of each teacher vs the true targets on training sequences (in-sample for old/big/in2) and on the 500 held-out sequences (honest for all). A big gap = in-sample optimism.

In [23]:
import numpy as np
from utils import weighted_pearson
from pipeline.data import SeqStore, split_train
st = SeqStore('datasets/train.parquet', 8)
train_ids, hold_ids = split_train(st.n_seqs, 500)
sample = {'train (300 seqs)': np.sort(np.random.default_rng(0).choice(train_ids, 300, replace=False)), 'holdout (500)': hold_ids}
Ys = {k: st.load(list(v))[1][:, 99:] for k, v in sample.items()}
for t in ['old', 'big', 'in2', 'oof']:
    T = np.load(f'/content/wnn/t_{t}.npy', mmap_mode='r')
    row = []
    for k, ids in sample.items():
        P = np.asarray(T[ids], dtype=np.float32)[:, 99:]
        row.append(f"{k}: {np.mean([weighted_pearson(Ys[k][..., j].ravel(), P[..., j].ravel()) for j in range(2)]):.4f}")
    print(f'{t:4s}', '   '.join(row))

old  train (300 seqs): 0.4912   holdout (500): 0.4607
big  train (300 seqs): 0.4900   holdout (500): 0.4614
in2  train (300 seqs): 0.4906   holdout (500): 0.4562
oof  train (300 seqs): 0.4579   holdout (500): 0.4534


## 6. Phase 3: students (5 passes; `st_big_e10` runs 10)

In [24]:
import json
experiments = [{'name': 'st_old', 'distill': 1.0, 'teacher': '/content/wnn/t_old.npy'},
 {'name': 'st_old_s1', 'distill': 1.0, 'teacher': '/content/wnn/t_old.npy', 'seed': 1},
 {'name': 'st_big', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy'},
 {'name': 'st_big_s1', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'seed': 1},
 {'name': 'st_in2', 'distill': 1.0, 'teacher': '/content/wnn/t_in2.npy'},
 {'name': 'st_oof', 'distill': 1.0, 'teacher': '/content/wnn/t_oof.npy'},
 {'name': 'st_big_e10', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'epochs': 10},
 {'name': 'st_big_h96', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'hidden': 96},
 {'name': 'st_big_h160', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'hidden': 160},
 {'name': 'st_big_h192', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'hidden': 192},
 {'name': 'st_big_l3h96', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'layers': 3, 'hidden': 96},
 {'name': 'st_big_wnone', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'distill_weight': 'none'},
 {'name': 'st_big_wteacher',
  'distill': 1.0,
  'teacher': '/content/wnn/t_big.npy',
  'distill_weight': 'teacher'},
 {'name': 'st_big_lr1e-3', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'lr': 0.001},
 {'name': 'st_big_drop', 'distill': 1.0, 'teacher': '/content/wnn/t_big.npy', 'dropout': 0.1},
 {'name': 'st_big_d90', 'distill': 0.9, 'teacher': '/content/wnn/t_big.npy'}]
json.dump(experiments, open('experiments.json', 'w'), indent=1)
print(len(experiments), 'experiments')

16 experiments


In [25]:
SWEEP = f'{OUT_DIR}/sweep4'
!python -m pipeline.multi_train --config experiments.json --epochs 5 --evals-per-epoch 1 --threads 8 --cache-eval --out-dir {SWEEP} 2>&1 | grep -v '"event": "batch"'

[st_old] {"event": "start", "config": {"name": "st_old", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1, "lr": 0.002, "wd": 0.0001, "seed": 0, "cell": "gru", "chunk": null, "ema": 0.0, "distill": 1.0, "distill_weight": "y", "teacher": "/content/wnn/t_old.npy", "epochs": null, "fold": null}, "common": {"epochs": 5, "batch": 256, "chunk": 500, "clip_grad": 1.0, "train_seqs": null, "holdout": 500, "valid_seqs": 400, "evals_per_epoch": 1, "seed": 0, "device": "auto", "threads": 8, "cache_eval": true, "cache_train": false, "data_dir": "datasets", "out_dir": "/content/drive/MyDrive/wunder-connectome/runs/sweep4", "teacher": null, "n_folds": 4, "extra": {}}, "device": "cuda", "params": 192258, "cost": 1.0, "train_seqs": 10107, "holdout": 500, "steps": 8000}
[st_old_s1] {"event": "start", "config": {"name": "st_old_s1", "hidden": 128, "layers": 2, "diff": false, "dropout": 0.0, "init_baseline": false, "loss": "wmse", "alpha": 0.1

## 7. Results (`cost` = multiply-adds per row vs the baseline)
Sweep 3 reference: d100_h128 (teacher `old`) 0.6821 valid / 0.5136 valid_all / 0.4569 holdout.

In [26]:
import pandas as pd
rows = [json.loads(l) for l in open(f'{SWEEP}/summary.jsonl')]
df = pd.DataFrame(rows).drop_duplicates('name', keep='last')
df['select'] = (df.valid_scored + df.holdout_all) / 2
df.sort_values('select', ascending=False)

,name,params,cost,valid_scored,valid_all,holdout_all,t0,t1,select
0,st_big_e10,192258,1.00,0.682828,0.514900,0.459980,0.6784,0.6873,0.571404
1,st_big_h192,398978,2.08,0.682367,0.514255,0.459040,0.6783,0.6864,0.570704
2,st_big_h160,286402,1.49,0.682391,0.514319,0.459002,0.6784,0.6864,0.570696
3,st_big_s1,192258,1.00,0.681867,0.513456,0.458549,0.6777,0.6860,0.570208
4,st_big,192258,1.00,0.682289,0.514074,0.458104,0.6782,0.6864,0.570196
5,st_big_d90,192258,1.00,0.682377,0.514095,0.457979,0.6781,0.6866,0.570178
6,st_big_drop,192258,1.00,0.681807,0.513426,0.458373,0.6778,0.6858,0.570090
7,st_big_wnone,192258,1.00,0.682081,0.514204,0.457772,0.6780,0.6862,0.569927
8,st_old_s1,192258,1.00,0.682017,0.512762,0.457815,0.6780,0.6861,0.569916
9,st_big_wteacher,192258,1.00,0.682148,0.513814,0.457603,0.6778,0.6865,0.569875


## 8. Ensembles under cost budgets (students + everything else)

In [27]:
ENS = f'{OUT_DIR}/ensembles_sweep4'
!python -m pipeline.ensemble --ckpts "{SWEEP}/*/best.pt" "{OUT_DIR}/sweep3/*/best.pt" "{OUT_DIR}/sweep2/*/best.pt" --threads 8 --max-members 4 --budgets 1 1.5 2 --export {ENS} 2>&1 | grep -vE 'Warning|_generic_rnn|warnings.warn'

40 models: st_big, st_big_d90, st_big_drop, st_big_e10, st_big_h160, st_big_h192, st_big_h96, st_big_l3h96, st_big_lr1e-3, st_big_s1, st_big_wnone, st_big_wteacher, st_in2, st_old, st_old_s1, st_oof, chunk2000_h128, d100_h128, d50_ema_h128, d50_h128, d50_h128_s1, d50_h96, ema_h128, h64, h64_s1, h96, h96_s1, l1_h192, l3_h96, lr4e-3_h128, lstm_h112, r0_wmse_h128, c1_mix_diff_h128, c2_mix_diff_h128_drop, c3_mix_diff_h192_drop, c3b_mix_diff_h192_drop_s1, c4_mix_diff_h256_drop, s0_wmse_h128, s1_wmse_h128, s2_wmse_h128
predictions collected in 570s

combo                           n  cost  holdout valid_sc valid_all
st_big                          1  1.00  0.45810  0.68229   0.51407
st_big_d90                      1  1.00  0.45798  0.68238   0.51410
st_big_drop                     1  1.00  0.45837  0.68181   0.51343
st_big_e10                      1  1.00  0.45998  0.68283   0.51490
st_big_h160                     1  1.49  0.45900  0.68239   0.51432
st_big_h192                     1  2.08  0

## 9. Speed on this CPU vs the official baseline
Grader ≈ 0.52 × these times (baseline: 80 µs here, 41 µs there). Our v2 export took 28.5 min on the grader. Keep candidates at or below ~1.5x here.

In [28]:
from pathlib import Path
cands = [f'{SWEEP}/{n}/submission' for n in ['st_big', 'st_big_h96', 'st_big_h160', 'st_big_h192', 'st_big_l3h96']] +         sorted(str(p) for p in Path(ENS).glob('cost*/submission'))
!python -m pipeline.speed {' '.join(cands)}

baseline                                              82.0 us/row   1.00x baseline  ~ 53.8 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep4/st_big/submission    81.1 us/row   0.99x baseline  ~ 53.2 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep4/st_big_h96/submission    65.2 us/row   0.80x baseline  ~ 42.8 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep4/st_big_h160/submission   107.1 us/row   1.31x baseline  ~ 70.4 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep4/st_big_h192/submission   125.0 us/row   1.52x baseline  ~ 82.1 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/sweep4/st_big_l3h96/submission    87.4 us/row   1.07x baseline  ~ 57.4 min for the test set here
/content/drive/MyDrive/wunder-connectome/runs/ensembles_sweep4/cost1.5/submission    81.2 us/row   0.99x baseline  ~ 53.3 min for the test set here
/content/drive/MyDrive/wunder-

## 10. Upload-ready zips (solution.py at the zip root) in Drive `wunder-connectome/runs/zips/`

In [29]:
import shutil, pandas as pd
os.makedirs(f'{OUT_DIR}/zips', exist_ok=True)
top = df.sort_values('select', ascending=False).name.head(3).tolist()   # df from section 7
srcs = {f'sweep4_{n}': f'{SWEEP}/{n}/submission' for n in top}
srcs.update({f'ens4_{p.parent.name}': str(p) for p in Path(ENS).glob('cost*/submission')})
for name, src in srcs.items():
    print(shutil.make_archive(f'{OUT_DIR}/zips/{name}', 'zip', src))

/content/drive/MyDrive/wunder-connectome/runs/zips/sweep4_st_big_e10.zip
/content/drive/MyDrive/wunder-connectome/runs/zips/sweep4_st_big_h192.zip
/content/drive/MyDrive/wunder-connectome/runs/zips/sweep4_st_big_h160.zip
/content/drive/MyDrive/wunder-connectome/runs/zips/ens4_cost1.zip
/content/drive/MyDrive/wunder-connectome/runs/zips/ens4_cost1.5.zip
/content/drive/MyDrive/wunder-connectome/runs/zips/ens4_cost2.zip
